[//]: # (cr:doc name='cycle_overview' id=cycle-intro)
# Fix Cycle Validation (template)

One notebook per fix cycle. Copy to
`debug/<engagement>/cycles/NNN_<slug>.ipynb` and fill in:
1. `CYCLE_ID`, `CYCLE_SLUG`, `RUN_ID` in cell `cycle-config`.
2. `REQUIRED_CHECKS` — the probe-check labels that must PASS to close the cycle.
3. Any cycle-specific assertions in the bottom section.

This notebook **reads** the probe's `result.json` from the current run; it
does not re-run the full probe. Run the probe first (or re-run it as part of
the same deploy), then this notebook filters the checks the cycle cares about.


In [ ]:
# @cr:config name='cycle_config' id=cycle-config
CYCLE_ID = 0
CYCLE_SLUG = ""
RUN_ID = ""
EXPERIMENTS_SUBDIR = "experiments"

REQUIRED_CHECKS: list[tuple[str, str, str]] = [
    # (layer, dataset, check) — must appear in probe result.json with status=PASS.
    # ("landing", "<dataset>", "exists_nonzero"),
]


In [ ]:
# @cr:code name='init_progress' id=cycle-init
from customer_retention.analysis.notebook_progress import accept_workflow_params

accept_workflow_params()

import json
from pathlib import Path

from customer_retention.analysis.auto_explorer import RunNamespace, mark_notebook
from customer_retention.analysis.visualization import console, display_table
from customer_retention.core.compat import native_pd
from customer_retention.core.config.experiments import get_experiments_dir

_namespace = RunNamespace.from_env_or_latest(root=get_experiments_dir())
if RUN_ID:
    _namespace = RunNamespace(root=get_experiments_dir(), run_id=RUN_ID)
mark_notebook(_namespace, f"cycle_{CYCLE_ID:03d}_{CYCLE_SLUG}.ipynb")

PROBE_RESULT = Path(_namespace.session_dir) / "probe" / "result.json"
CYCLE_DIR = Path(_namespace.session_dir) / f"cycle_{CYCLE_ID:03d}"
CYCLE_DIR.mkdir(parents=True, exist_ok=True)

# --- cr:profiler ---
if __import__('os').environ.get("CR_BATCH_EXECUTION") == "1":
    import json as _j
    import os as _os
    import re as _r
    _cr_nb = _os.path.splitext(_os.path.basename(_os.environ.get("PAPERMILL_OUTPUT_PATH", "")))[0]
    if _cr_nb:
        _cr_mp = _os.path.join(_os.getcwd(), f".cr_cell_metrics_{_cr_nb}.jsonl")
        open(_cr_mp, 'w').close()
        _cr_re = _r.compile(r"^#\s*@cr:\w+\s+name='([^']+)'\s+id=(\w+)")
        def _cr_jc():
            return -1
        try:
            _s = __import__('pyspark.sql', fromlist=['SparkSession']).SparkSession.getActiveSession()
            if _s:
                def _cr_jc():  # noqa: F811
                    return _s._jsc.sc().dagScheduler().nextJobId().get()
        except Exception:
            pass
        def _cr_pre(info):
            info._cr_sj = _cr_jc()
        def _cr_post(r):
            sj = getattr(r.info, '_cr_sj', -1)
            sa = _cr_jc()
            m = _cr_re.match((r.info.raw_cell or '').split('\n')[0])
            if m:
                with open(_cr_mp, 'a') as f:
                    f.write(_j.dumps({"cell_name": m.group(1), "cell_id": m.group(2),
                                      "spark_jobs": (sa - sj) if sj >= 0 and sa >= 0 else None}) + '\n')
        get_ipython().events.register('pre_run_cell', _cr_pre)
        get_ipython().events.register('post_run_cell', _cr_post)
# --- /cr:profiler ---


In [ ]:
# @cr:code name='cycle_load_probe' id=cycle-load
if not PROBE_RESULT.exists():
    raise FileNotFoundError(
        f"probe result missing at {PROBE_RESULT} — run probe_template.ipynb first"
    )
PROBE = json.loads(PROBE_RESULT.read_text())
CHECKS = PROBE["checks"]
console.print(f"loaded {len(CHECKS)} probe checks from {PROBE_RESULT}")
console.print(f"probe overall status: {PROBE['status']}  ({PROBE['passed']}/{PROBE['total']} passed)")


In [ ]:
# @cr:code name='cycle_filter_required' id=cycle-filter
rows = []
missing = []
for layer, dataset, check in REQUIRED_CHECKS:
    hit = next((c for c in CHECKS if c["layer"] == layer and c["dataset"] == dataset and c["check"] == check), None)
    if hit is None:
        missing.append((layer, dataset, check))
        rows.append({"layer": layer, "dataset": dataset, "check": check,
                     "status": "MISSING", "detail": "check not found in probe result"})
    else:
        rows.append(hit)

df = native_pd.DataFrame(rows)
display_table(df)

closed = bool(rows) and not missing and all(r["status"] == "PASS" for r in rows)
console.print(f"cycle {CYCLE_ID:03d} gate: {'CLOSED' if closed else 'OPEN'}  "
              f"({sum(1 for r in rows if r['status'] == 'PASS')}/{len(REQUIRED_CHECKS)} required passing)")


[//]: # (cr:doc name='cycle_custom_assertions' id=cycle-custom-md)
## Cycle-specific assertions (optional)

Add code cells here for any check that the universal probe does not cover —
e.g. "the generated bronze spec contains literal `per_grid_date_mode=True`",
"the findings YAML has a `milestone_pairs` field for dataset X". Each cell
appends to `rows` so the final summary captures them.


In [ ]:
# @cr:code name='cycle_result' id=cycle-result
result = {
    "cycle": CYCLE_ID,
    "slug": CYCLE_SLUG,
    "run_id": _namespace.run_id,
    "status": "PASS" if closed else "FAIL",
    "required": len(REQUIRED_CHECKS),
    "checks": rows,
}
(CYCLE_DIR / "result.json").write_text(json.dumps(result, indent=2, default=str))
console.print(f"result -> {CYCLE_DIR / 'result.json'}")
